In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

df = pd.read_csv("cleaned_dataset.csv")

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["month"] = df["Date"].dt.month

target = "Retard moyen de tous les trains à l'arrivée"

features = [
    "month",
    "Durée moyenne du trajet",
    "Nombre de circulations prévues",
    "Service",
    "Gare de départ",
    "Gare d'arrivée"
]

X = df[features].copy()
y = df[target]

X = pd.get_dummies(
    X,
    columns=["Service", "Gare de départ", "Gare d'arrivée"],
    drop_first=False
)

X = X.replace([np.inf, -np.inf], np.nan)
X = X.dropna()
y = y.loc[X.index]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = HistGradientBoostingRegressor(
    max_iter=600,
    learning_rate=0.05,
    max_depth=6,
    min_samples_leaf=20,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Test R2:", r2_score(y_test, y_pred))
print("Test MAE:", mean_absolute_error(y_test, y_pred))

cv_scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="r2"
)

print("CV R2 mean:", cv_scores.mean())



Test R2: 0.43920220418813916
Test MAE: 2.160798349442033
CV R2 mean: 0.2723902529408787
